# german manufacturing industry extractor functions

## Introduction

This notebook contains two Python functions that extract data from the German GENESIS-Online database via API requests, clean and transform the data, calculate additional columns, and store the processed datasets in the user's MySQL database.

## Requirements

The user needs a GENESIS-Online account consisting of an email address (used as the username) and a password. To store the data, the user also needs their MySQL connection details: host, username, port and password.

If the user has previously used either function, the person also needs to know the names of the existing database and table so they can provide this information to the functions and store the additional data there.

## Database

Data Sources

The notebook uses two tables from the German GENESIS-Online database:

Function_monthly() uses Table 42111-0026 – Monthly data: Contains monthly data on employment and turnover by economic sector. The original columns include the reporting period, industry classification, statistical indicator, and value. The data covers topics such as total turnover, domestic and foreign turnover, employment, and the number of reporting units.

Function_annual() uses Table 48121-0003 – Annual data: Contains annual business statistics by economic sector and company size. The data covers the number of enterprises, turnover, persons employed, gross investment in tangible fixed assets, and gross value added.

Both tables provide the basis for analyzing developments in the German manufacturing industry across different economic sectors.

The following manufacturing sectors and codes are available. For the functions to work, you have to enter the WZ code, not the sector name. You can find a list of the available sectors in the PDF provided in the repository. This list is based on the code list provided by GENESIS. Please keep in mind that codes may change or be deleted by the Federal Statistical Office. Always check after each function run whether the data was stored successfully. 

Besides the data and the API-request codes, GENESIS also provided two functions to properly unzip and convert the downloaded data into a dataframe. These two functions are integreted into my functions and not my creation: 


    def csvUnZip():
        filebytes = io.BytesIO(.content)
        zipFile = zipfile.ZipFile(filebytes)
        csvFile = zipFile.open(zipFile.namelist()[0])
        return csvFile


    def readCsv(csvFile):
        df = pd.read_csv(csvFile, delimiter=';', decimal=",", na_values=["...",".","-","/","x"])
        return df


## Function Descriptions

### Runtime
Please note that after entering all required information the function runs for multiple minutes, in my cases usually around 5 Minutes. This is mostly caused by the API request. After asking for the Downloadcode, my function immediately receives it. But it takes time for the request to get stored in the GENESIS server. For that reason, a timer of 200 seconds is integreted. After receiving the necessary download code, the function stops for 200 seconds which proved to be enough time for the request being stored within the GENESIS server.

In this chapter, the functionality of each function will be explained.

### Function_monthly()
1. User Inputs:

At the beginning, the function asks the user to provide the following information:

- Industry Code: The WZ code of the industry to be extracted from GENESIS-Online. For example: WZ08-109

- Start year: The first year of the period for which monthly data should be downloaded.

- End year: The last year of the period for which monthly data should be downloaded.

- GENESIS username: The username required to access the GENESIS-Online API.

- GENESIS password: The password required to authenticate with GENESIS-Online. The password is entered using getpass() and is therefore not displayed in the console.

Following that, the function asks whether the user wants to add the data to an already existing database and table in case the function has already been used:

- Existing database (yes / no): Determines whether the function should use an existing database or create a new one.

If the user enters yes:

- Database name: The exact name of the existing MySQL database must be entered.

- Existing table (yes / no): Determines whether the user has already used this function to create the table in which the data should be stored.

- Table name: If the table already exists, the user must enter its name.

If the user enters no:

- Database name: The user defines a name for the new database. The function creates the database and a table named monthly_industry_data.


The function then requests the connection details required to access the MySQL server:

- Host: The hostname or IP address of the MySQL server.

- SQL user: The username used to connect to the MySQL server.

- SQL password: The password for the MySQL account. It is entered using getpass() and is not displayed in the console.

- Port: The port used to connect to the MySQL server.

These details are used to establish the connection and store the processed data in the selected database and table.


2. Data Extraction and Processing

- Requests monthly employment and turnover data from GENESIS-Online via its API for the selected period.

- Downloads the requested data, extracts the CSV file from the ZIP archive, and loads it into a pandas DataFrame.

- Removes unnecessary columns and renames the remaining columns.

- Converts month codes into numeric month values.

- If a new database is created, calculates the total manufacturing industry figures and adds them under the code WZ08-TOTAL.

- Filters the data according to the selected industry code.

- Converts turnover values from thousands of euros to euros.

- Restructures the data so that each metric has its own column.

- Calculates additional indicators, including turnover shares, monthly turnover per employee, and employees per survey unit.

- Organizes the columns into a predefined order.

- Connects to MySQL and creates the required database and table if necessary, or appends the data to an existing table.

- Returns the processed DataFrame.

## How to use the functions:

Always execute import section first. If neccessary, install the requiered packages. 

Run def Funktion_monthly(): and def Funktion_annual():  

After that, execute the chunk consisting df_Empl_Reven_cleaned = Funktion_monthly() or Industry_by_Size = Funktion_annual(). 

In [1]:
import pandas as pd
import requests
import io
import zipfile
import time
from getpass import getpass
from sqlalchemy import create_engine, text

In [2]:
def Funktion_monthly():
    Industry_Code = input("Please type in Industry Code: ")
    Startdate = input("Please enter start year: ")
    Enddate = input("Please enter end year: ")
    username = input("GENESIS username: ")
    Gen_pass = getpass("GENESIS password: ")
    existing_database = input("Do you want to add the data to an existing database? (yes / no): ").strip().lower()

    if existing_database == "yes":
        database_exist = input("Please type in existing database name: ")
        table_exist = input("Have you already used this function and created this table? (yes / no): ").strip().lower()
        if table_exist == "yes":
            table_name = input("Please type in table name: ")
    else: 
        database_name = input("Please Define database name: ")

    host = input("Host: ")
    sql_user = input("SQL user: ")
    sql_pass = getpass("SQL password: ")
    port = input("Port: ")
   

    BASE_URL = "https://genesis.destatis.de/genesisWS/rest/2020/"

    headers_job = {
        "Content-Type": "application/x-www-form-urlencoded",
        "username": username,
        "password": Gen_pass,
    }

    #Asking for download code
    Employment_Revenue_table = requests.post(BASE_URL + "data/tablefile", headers = headers_job, data = {
        "name" : "42111-0026",
        "startyear" : Startdate,
        "endyear" : Enddate,
        "compress" : "true",
        "format" : "ffcsv",
        "language" : "en",
        "job" : "true"
    })

    time.sleep(200)

    code = Employment_Revenue_table.json()["Status"]["Content"]
    code = code.split()[-1]

    #Downloading the data
    Employment_Revenue_table = requests.post(BASE_URL + "data/resultfile", headers = headers_job, data = {
        "name" : code,
        "startyear" : Startdate,
        "endyear" : Enddate,
        "compress" : "true",
        "format" : "ffcsv", # ff better for use with pandas according to GENESIS
        "language" : "en",
        "job" : "true"
    })


    # implementing predefined function from GENESIS
    def csvUnZip(Employment_Revenue_table):
        filebytes = io.BytesIO(Employment_Revenue_table.content)
        zipFile = zipfile.ZipFile(filebytes)
        csvFile = zipFile.open(zipFile.namelist()[0])
        return csvFile

    Employment_Revenue_table = csvUnZip(Employment_Revenue_table)

    def readCsv(csvFile):
        df = pd.read_csv(csvFile, delimiter=';', decimal=",", na_values=["...",".","-","/","x"])
        return df

    df_Empl_Reven = readCsv(Employment_Revenue_table)

    # Deleting useless columns
    df_Empl_Reven_cleaned = df_Empl_Reven.drop([
        "statistics_code", "statistics_label", "time_code", "time_label",
        "1_variable_code", "2_variable_code", "2_variable_label",
        "2_variable_attribute_label", "3_variable_code", "3_variable_label", "3_variable_attribute_label",
        "4_variable_code", "4_variable_label", "2_variable_attribute_code",
        "3_variable_attribute_code"
        ], axis = 1)


    #renaming columns:
    df_Empl_Reven_cleaned = df_Empl_Reven_cleaned.rename(columns={
        "time" : "year",
        "1_variable_attribute_code" : "Month_Number",
        "1_variable_attribute_label" : "Month",
        "4_variable_attribute_code" : "industry_code",
        "4_variable_attribute_label" : "industry",
        "value_variable_code" : "metric_code",
        "value_variable_label" : "metric"
    })

    #Changing Month_Number format
    df_Empl_Reven_cleaned["Month_Number"] = df_Empl_Reven_cleaned["Month_Number"].str.replace("MONAT" , "")

    # Calculate total manufacturing industry for a new database
    if existing_database == "no":
        df_check = df_Empl_Reven_cleaned[
        df_Empl_Reven_cleaned["industry_code"].str.fullmatch(r"WZ08-(1\d|2\d|3[0-3])")
        ].copy()

        df_manufacturing_total = df_check.groupby(
        ["year", "Month_Number", "Month", "metric", "value_unit"], as_index=False
        )["value"].sum()

        df_manufacturing_total["industry_code"] = "WZ08-TOTAL"
        df_manufacturing_total["industry"] = "Total manufacturing industry"

        df_Empl_Reven_cleaned = pd.concat(
        [df_Empl_Reven_cleaned, df_manufacturing_total], ignore_index=True
        )

        # Keep the requested industry and the total
        df_Empl_Reven_cleaned = df_Empl_Reven_cleaned[
        df_Empl_Reven_cleaned["industry_code"].isin([Industry_Code, "WZ08-TOTAL"])
        ].copy()

    # Keep only the requested industry
    else:

        df_Empl_Reven_cleaned = df_Empl_Reven_cleaned[
        df_Empl_Reven_cleaned["industry_code"] == Industry_Code
        ].copy()

    #Final cleaning
    df_Empl_Reven_cleaned.loc[df_Empl_Reven_cleaned["value_unit"] == "EUR 1000", "value"] *= 1000
    df_Empl_Reven_cleaned = df_Empl_Reven_cleaned.drop(columns=["value_unit", "metric_code", "1_variable_label"])

    #building additional columns out of the metric column
    df_Empl_Reven_cleaned = df_Empl_Reven_cleaned.pivot(
    index=["year", "Month_Number", "Month", "industry_code", "industry"],
    columns="metric",
    values="value"
    ).reset_index()

    df_Empl_Reven_cleaned.columns.name = None

    #renaming new columns
    df_Empl_Reven_cleaned = df_Empl_Reven_cleaned.rename(columns={
        "Domestic turnover" : "Domestic_turnover",
        "Non-domestic turnover" : "Non_domestic_turnover",
        "Other non-domestic turnover" : "Other_non_domestic_turnover",
        "Persons employed" : "Persons_employed",
        "Survey units" : "Survey_units",
        "Turnover with euro-zone countries" : "Turnover_with_euro_zone_countries"
    })

    # Building additional columns
    df_Empl_Reven_cleaned["Employees_per_survey_unit"] = df_Empl_Reven_cleaned["Persons_employed"] / df_Empl_Reven_cleaned["Survey_units"]
    df_Empl_Reven_cleaned["Monthly_turnover_per_employee"] = df_Empl_Reven_cleaned["Turnover"] / df_Empl_Reven_cleaned["Persons_employed"]
    df_Empl_Reven_cleaned["Monthly_domestic_turnover_share"] = (df_Empl_Reven_cleaned["Domestic_turnover"] / df_Empl_Reven_cleaned["Turnover"]) * 100
    df_Empl_Reven_cleaned["Monthly_EU_turnover_share"] = (df_Empl_Reven_cleaned["Turnover_with_euro_zone_countries"] / df_Empl_Reven_cleaned["Turnover"]) * 100
    df_Empl_Reven_cleaned["Monthly_other_non_domestic_turnover_share"] = (df_Empl_Reven_cleaned["Other_non_domestic_turnover"] / df_Empl_Reven_cleaned["Turnover"]) * 100
    df_Empl_Reven_cleaned["Monthly_non_domestic_turnover_share"] = (df_Empl_Reven_cleaned["Non_domestic_turnover"] / df_Empl_Reven_cleaned["Turnover"]) * 100

    # Final structuring

    df_Empl_Reven_cleaned = df_Empl_Reven_cleaned[[
        "year", "Month_Number", "Month",
        "industry_code", "industry", "Turnover", "Domestic_turnover",
        "Monthly_domestic_turnover_share", "Non_domestic_turnover", "Monthly_non_domestic_turnover_share",
        "Other_non_domestic_turnover", "Monthly_other_non_domestic_turnover_share",
        "Persons_employed", "Monthly_turnover_per_employee", "Survey_units", "Employees_per_survey_unit",
        "Turnover_with_euro_zone_countries", "Monthly_EU_turnover_share"
    ]]

    #Connecting to SQL
    server_engine = create_engine(f"mysql+pymysql://{sql_user}:{sql_pass}@{host}:{port}/")

    #SQL activity based on user decisions
    if existing_database == "no" or (existing_database == "yes" and table_exist == "no"):
        if existing_database == "no":
            with server_engine.begin() as conn:
                conn.execute(text(f"CREATE DATABASE `{database_name}`"))
            connection_string = f"mysql+pymysql://{sql_user}:{sql_pass}@{host}:{port}/{database_name}"
        else:
            connection_string = f"mysql+pymysql://{sql_user}:{sql_pass}@{host}:{port}/{database_exist}" 

        engine = create_engine(connection_string)

        #Creating SQL Table
        with engine.begin() as conn:
            conn.execute(text("""
                CREATE TABLE monthly_industry_data (
                    year INT,
                    Month_Number INT,
                    Month VARCHAR(30),
                    industry_code VARCHAR(30),
                    industry VARCHAR(260),
                    Turnover BIGINT,
                    Domestic_turnover BIGINT,
                    Monthly_domestic_turnover_share DOUBLE,
                    Non_domestic_turnover BIGINT,
                    Monthly_non_domestic_turnover_share DOUBLE,
                    Other_non_domestic_turnover BIGINT,
                    Monthly_other_non_domestic_turnover_share DOUBLE,
                    Persons_employed BIGINT,
                    Monthly_turnover_per_employee DOUBLE,
                    Survey_units INT,
                    Employees_per_survey_unit DOUBLE,
                    Turnover_with_euro_zone_countries BIGINT,
                    Monthly_EU_turnover_share DOUBLE
                )
            """))

        df_Empl_Reven_cleaned.to_sql("monthly_industry_data", con=engine, if_exists="append", index=False)

    else: 
        schema = database_exist
        connection_string = f"mysql+pymysql://{sql_user}:{sql_pass}@{host}:{port}/{database_exist}"
        engine = create_engine(connection_string)
        df_Empl_Reven_cleaned.to_sql(table_name, con=engine, if_exists="append", index=False)


    return df_Empl_Reven_cleaned

In [ ]:
df_Empl_Reven_cleaned = Funktion_monthly()

: 

### Function_annual()


### Runtime
Please note that after entering all required information the function runs for multiple minutes, in my cases usually around 5 Minutes. This is mostly caused by the API request. After asking for the Downloadcode, my function immediately receives it. But it takes time for the request to get stored in the GENESIS server. For that reason, a timer of 200 seconds is integreted. After receiving the necessary download code, the function stops for 200 seconds which proved to be enough time for the request being stored within the GENESIS server.


1. User Inputs

At the beginning, the function asks the user to provide the following information:

- Industry Code: The WZ code of the industry to be extracted from GENESIS-Online. For example: WZ08-109

- Start year: The first year of the period for which annual data should be downloaded.

- End year: The last year of the period for which annual data should be downloaded.

- GENESIS username: The username required to access the GENESIS-Online API.

- GENESIS password: The password required to authenticate with GENESIS-Online. The password is entered using getpass() and is therefore not displayed in the console.


Following that, the function asks whether the user wants to add the data to an already existing database and table in case the function has already been used:

- Existing database (yes / no): Determines whether the function should use an existing database or create a new one.

- If the user enters yes:

- Database name: The exact name of the existing MySQL database must be entered.

- Existing table (yes / no): Determines whether the user has already used this function to create the table in which the data should be stored.

- Table name: If the table already exists, the user must enter its name.


If the user enters no:

- Database name: The user defines a name for the new database. The function creates the database and a table named annual_industry_data.


The function then requests the connection details required to access the MySQL server:

- Host: The hostname or IP address of the MySQL server.

- SQL user: The username used to connect to the MySQL server.

- SQL password: The password for the MySQL account. It is entered using getpass() and is not displayed in the console.

- Port: The port used to connect to the MySQL server.

These details are used to establish the connection and store the processed data in the selected database and table.


2. Data Extraction and Processing

- Requests annual business statistics from GENESIS-Online via its API for the selected period using Table 48121-0003.

- Downloads the requested data, extracts the CSV file from the ZIP archive, and loads it into a pandas DataFrame.

- Removes unnecessary columns and renames the remaining columns.

- Converts turnover and other monetary values from millions of euros to euros where the original unit is EUR mn.

- Restructures the data so that each statistical indicator has its own column.

- Cleans the column names by replacing special characters and spaces with underscores.

- Filters the data according to the selected industry code.

- Connects to MySQL and creates the required database and table if necessary, or appends the data to an existing table.

Returns the processed DataFrame.

In [3]:
def Funktion_annual():
    Industry_Code = input("Please type in Industry Code: ")
    Startdate = input("Please enter start year: ")
    Enddate = input("Please enter end year: ")
    username = input("GENESIS username: ")
    Gen_pass = getpass("GENESIS password: ")
    existing_database = input("Do you want to add the data to an existing database? (yes / no): ").strip().lower()

    if existing_database == "yes":
        database_exist = input("Please type in existing database name: ")
        table_exist = input("Have you already used this function and created this table? (yes / no): ").strip().lower()
        if table_exist == "yes":
            table_name = input("Please type in table name: ")
    else: 
        database_name = input("Please Define database name: ")

    host = input("Host: ")
    sql_user = input("SQL user: ")
    sql_pass = getpass("SQL password: ")
    port = input("Port: ")

    BASE_URL = "https://genesis.destatis.de/genesisWS/rest/2020/"

    headers_job = {
    "Content-Type": "application/x-www-form-urlencoded",
    "username": username,
    "password": Gen_pass,
    }

    Industry_by_Size = requests.post(BASE_URL + "data/tablefile", headers = headers_job, data = {
    "name" : "48121-0003",
    "startyear" : Startdate,
    "endyear" : Enddate,
    "compress" : "true",
    "format" : "ffcsv",
    "language" : "en",
    "job" : "true"
    })

    time.sleep(200)

    code = Industry_by_Size.json()["Status"]["Content"]
    code = code.split()[-1]

    #Downloading the data
    Industry_by_Size_csv = requests.post(BASE_URL + "data/resultfile", headers = headers_job, data = {
    "name" : code,
    "startyear" : Startdate,
    "endyear" : Enddate,
    "compress" : "true",
    "format" : "ffcsv", # ff better for use with pandas according to GENESIS
    "language" : "en",
    "job" : "true"
    })

    # implementing predefined function from GENESIS
    def csvUnZip(Employment_Revenue_table):
        filebytes = io.BytesIO(Employment_Revenue_table.content)
        zipFile = zipfile.ZipFile(filebytes)
        csvFile = zipFile.open(zipFile.namelist()[0])
        return csvFile

    Industry_by_Size_csv = csvUnZip(Industry_by_Size_csv)

    def readCsv(csvFile):
        df = pd.read_csv(csvFile, delimiter=';', decimal=",", na_values=["...",".","-","/","x"])
        return df

    Industry_by_Size_csv = readCsv(Industry_by_Size_csv)

    Industry_by_Size = Industry_by_Size_csv.drop([
        "statistics_code", "statistics_label",
        "time_code", "time_label",
        "1_variable_code", "1_variable_label",
        "1_variable_attribute_code", "1_variable_attribute_label", "2_variable_code",
        "2_variable_label", #"2_variable_attribute_code",
        "3_variable_label", "3_variable_code",
        "value_variable_code"
        ], axis = 1)

    Industry_by_Size.loc[Industry_by_Size["value_unit"] == "EUR mn", "value"] *= 1000000
    Industry_by_Size = Industry_by_Size.drop("value_unit", axis= 1)

    Industry_by_Size = Industry_by_Size.rename(columns={
    "time" : "year",
    "2_variable_attribute_label" : "Company_Size",
    "3_variable_attribute_label" : "Industry",
    "3_variable_attribute_code" : "industry_Code"
    })

    Industry_by_Size = Industry_by_Size.pivot(
    index=["year", "Company_Size", "Industry", "industry_Code"],
    columns="value_variable_label",
    values="value"
    ).reset_index()
    Industry_by_Size.columns.name = None


    Industry_by_Size.columns = Industry_by_Size.columns.str.replace(r"[^A-Za-z0-9_]+", "_", regex=True).str.strip("_")

    Industry_by_Size = Industry_by_Size[Industry_by_Size["industry_Code"] == Industry_Code]


    #Connecting to SQL
    server_engine = create_engine(f"mysql+pymysql://{sql_user}:{sql_pass}@{host}:{port}/")

    #SQL activity based on user decisions
    if existing_database == "no" or (existing_database == "yes" and table_exist == "no"):
        if existing_database == "no":
            with server_engine.begin() as conn:
                conn.execute(text(f"CREATE DATABASE `{database_name}`")) 
        
        if existing_database == "no":
            connection_string = f"mysql+pymysql://{sql_user}:{sql_pass}@{host}:{port}/{database_name}"
        else:
            connection_string = f"mysql+pymysql://{sql_user}:{sql_pass}@{host}:{port}/{database_exist}"

        engine = create_engine(connection_string)

        # building table
        with engine.begin() as conn:
            conn.execute(text("""
            CREATE TABLE annual_industry_data (
                year INT,
                Company_Size VARCHAR(100),
                Industry VARCHAR(255),
                industry_Code VARCHAR(30),
                Changes_in_stock_of_goods BIGINT,
                Cost_of_social_insurances BIGINT,
                Costs_for_rents_and_leases BIGINT,
                Employees BIGINT,
                Employees_in_full_time_equivalents BIGINT,
                Enterprises_EU BIGINT,
                Enterprises_with_investments BIGINT,
                Expenditure_on_temporary_employees BIGINT,
                Gross_invest_in_tangible_fixed_assets_p_pers_empl BIGINT,
                Gross_investment_in_tangible_fixed_assets BIGINT,
                Gross_operating_surplus BIGINT,
                Gross_operating_surplus_per_person_employed BIGINT,
                Gross_value_added_at_factor_cost BIGINT,
                Gross_value_added_at_factor_cost_p_pers_employed BIGINT,
                Gross_wages_and_salaries BIGINT,
                Persons_employed BIGINT,
                Production_value BIGINT,
                Purchases_of_goods_a_services_for_resale_i_s_con BIGINT,
                Purchases_of_goods_and_services BIGINT,
                Staff_expenditure BIGINT,
                Staff_expenditure_per_person_employed BIGINT,
                Turnover BIGINT,
                Turnover_per_person_employed BIGINT
            )
            """))

        Industry_by_Size.to_sql("annual_industry_data", con=engine, if_exists="append", index=False)

    else:
        schema = database_exist
        connection_string = f"mysql+pymysql://{sql_user}:{sql_pass}@{host}:{port}/{database_exist}"
        engine = create_engine(connection_string)
        Industry_by_Size.to_sql(table_name, con=engine, if_exists="append", index=False) 
    

    return Industry_by_Size

In [8]:
Industry_by_Size = Funktion_annual()